# Model Comparison & Training
Comparing Random Forest and XGBoost on real enterprise data. Handles NaN from departmental aggregation join.

In [1]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, precision_recall_fscore_support
import joblib
import json
import os
from datetime import datetime

df = pd.read_csv('../data/processed/features/attrition_features.csv')
X = df.drop(columns=['Attrition_encoded'])
y = df['Attrition_encoded']

# Fill NaN from dept-level aggregation join misses
nan_count = X.isnull().sum().sum()
if nan_count > 0:
    X = X.fillna(X.median(numeric_only=True))
print(f'Feature matrix: {X.shape} | NaN filled: {nan_count} | Remaining: {X.isnull().sum().sum()}')
print(f'Target: {y.value_counts().to_dict()}')

Feature matrix: (500, 42) | NaN filled: 0 | Remaining: 0
Target: {0: 445, 1: 55}


In [2]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
pos_weight = (len(y) - sum(y)) / max(sum(y), 1)
print(f'Train: {len(X_train)} | Test: {len(X_test)} | scale_pos_weight: {pos_weight:.2f}')

models = {
    'Random Forest': RandomForestClassifier(n_estimators=200, random_state=42, class_weight='balanced', n_jobs=-1),
    'XGBoost': XGBClassifier(n_estimators=300, random_state=42, scale_pos_weight=pos_weight,
                              learning_rate=0.05, max_depth=6, eval_metric='logloss', verbosity=0)
}

results = []
best_recall = 0
best_model_name = ''
best_model = None

for name, model in models.items():
    print(f'Training {name}...')
    model.fit(X_train, y_train)
    probs = model.predict_proba(X_test)[:, 1]
    preds = model.predict(X_test)
    precision, recall, f1, _ = precision_recall_fscore_support(y_test, preds, average='binary', zero_division=0)
    roc_auc = roc_auc_score(y_test, probs)
    results.append({'Model': name, 'ROC-AUC': round(roc_auc, 4), 'F1': round(f1, 4), 'Recall': round(recall, 4), 'Precision': round(precision, 4)})
    print(f'  ROC-AUC: {roc_auc:.4f} | F1: {f1:.4f} | Recall: {recall:.4f}')
    if recall > best_recall:
        best_recall = recall
        best_model_name = name
        best_model = model

import pandas as pd
results_df = pd.DataFrame(results)
print()
print(results_df.to_string(index=False))
print(f'Winner: {best_model_name} (Recall={best_recall:.4f})')

Train: 400 | Test: 100 | scale_pos_weight: 8.09
Training Random Forest...


  ROC-AUC: 1.0000 | F1: 0.9524 | Recall: 0.9091
Training XGBoost...


  ROC-AUC: 1.0000 | F1: 1.0000 | Recall: 1.0000

        Model  ROC-AUC     F1  Recall  Precision
Random Forest      1.0 0.9524  0.9091        1.0
      XGBoost      1.0 1.0000  1.0000        1.0
Winner: XGBoost (Recall=1.0000)


In [3]:
os.makedirs('../models/v1', exist_ok=True)
joblib.dump(best_model, '../models/v1/attrition_pipeline.joblib')
joblib.dump(X.columns.tolist(), '../models/v1/feature_names.joblib')

best_row = results_df[results_df['Model'] == best_model_name].iloc[0]
metadata = {
    'model_name': 'Attrition Prediction Model',
    'version': 'v2.0',
    'algorithm': best_model_name,
    'training_date': datetime.now().strftime('%Y-%m-%d'),
    'join_strategy': 'Departmental aggregation (IDs incompatible across datasets)',
    'training_data': 'employee_performance_pro.csv + dept-agg from Employee_Performance_Dataset + Cleaned_HR_Data_Analysis',
    'n_training_samples': int(len(X_train)),
    'n_features': int(X.shape[1]),
    'roc_auc': float(best_row['ROC-AUC']),
    'f1_score': float(best_row['F1']),
    'recall': float(best_row['Recall']),
    'top_features_by_correlation': ['PerformanceRating', 'OvertimeHoursPerMonth', 'WorkLifeBalance', 'Age', 'YearsAtCompany'],
    'metrics': {'total_predictions_logged': 0}
}
with open('../models/v1/metadata.json', 'w') as f:
    json.dump(metadata, f, indent=4)

print(f'Model saved: {best_model_name}')
print(f'ROC-AUC: {best_row["ROC-AUC"]} | F1: {best_row["F1"]} | Recall: {best_row["Recall"]}')

Model saved: XGBoost
ROC-AUC: 1.0 | F1: 1.0 | Recall: 1.0
